# PMM Dynamic Screener — NonKYC Public REST

This notebook screens **NONKYC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

By default this notebook screens **all quote assets** available on NonKYC (USDT, XMR, BTC, USDC, etc.). To restrict to a single quote, set `QUOTE_ASSET` to e.g. `'USDT'`. To screen a specific set, use comma-separated values like `'USDT,XMR'`. The notebook uses the documented public REST endpoints for markets, tickers, order books, candles, and trades.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [1]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import NonKYCPublicScreener, default_nonkyc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.nonkyc_public import NONKYC_BASE_URL


PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "*"
INTERVAL = "5m"
UNIVERSE_TOP_K = 3000
FINAL_TOP_N = 15
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "nonkyc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_nonkyc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 50000.0
cfg.max_spread_bps = 120.0
cfg.min_top_of_book_quote = 5.0
cfg.min_depth_10bps_quote = 0.0
cfg.max_last_trade_age_sec = 3600.0
cfg.min_recent_trade_count = 40
cfg.min_candle_count = 220
cfg.min_candle_coverage_ratio = 0.9
cfg.max_zero_volume_fraction = 0.3
cfg.min_natr_bps = 12.0
cfg.max_natr_bps = 400.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


Screener config


,value
connector,nonkyc
quote_asset,*
interval,5m
universe_top_k,3000
final_top_n,15
candle_limit,288
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.2
timeout_seconds,30.0


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [3]:
screener = NonKYCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,nonkyc,*,5m,343,343


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,LTC-USDT,LTC/USDT,3.265157e+05,1.872484,78.015563,1.000000e-02,0.000100,active
1,XMR-USDT,XMR/USDT,1.501422e+06,29.753982,77.546947,1.000000e-02,0.001000,active
2,ARRR-USDT,ARRR/USDT,2.554688e+05,10.377127,77.266978,1.000000e-06,0.000100,active
3,TRX-USDT,TRX/USDT,2.141502e+05,15.905837,76.812511,1.000000e-04,0.010000,active
4,NKYC-USDT,NKYC/USDT,1.498705e+05,14.371827,75.923941,1.000000e-06,0.000100,active
5,MANA-USDT,MANA/USDT,7.653070e+04,8.038124,75.316198,1.000000e-05,0.010000,active
6,BTC-USDT,BTC/USDT,3.512106e+06,50.061071,75.220588,1.000000e-02,0.000001,active
7,W-USDT,W/USDT,3.516483e+05,37.664783,75.161207,1.000000e-05,0.010000,active
8,USDC-USDT,USDC/USDT,1.474665e+06,49.002450,74.960127,1.000000e-04,0.010000,active
9,SEI-USDT,SEI/USDT,1.577249e+05,36.630037,74.112288,1.000000e-04,0.010000,active


Shortlist for detailed enrichment: 343


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,LTC-USDT,LTC/USDT,3.265157e+05,1.872484,78.015563,1.000000e-02,0.000100,active
1,XMR-USDT,XMR/USDT,1.501422e+06,29.753982,77.546947,1.000000e-02,0.001000,active
2,ARRR-USDT,ARRR/USDT,2.554688e+05,10.377127,77.266978,1.000000e-06,0.000100,active
3,TRX-USDT,TRX/USDT,2.141502e+05,15.905837,76.812511,1.000000e-04,0.010000,active
4,NKYC-USDT,NKYC/USDT,1.498705e+05,14.371827,75.923941,1.000000e-06,0.000100,active
5,MANA-USDT,MANA/USDT,7.653070e+04,8.038124,75.316198,1.000000e-05,0.010000,active
6,BTC-USDT,BTC/USDT,3.512106e+06,50.061071,75.220588,1.000000e-02,0.000001,active
7,W-USDT,W/USDT,3.516483e+05,37.664783,75.161207,1.000000e-05,0.010000,active
8,USDC-USDT,USDC/USDT,1.474665e+06,49.002450,74.960127,1.000000e-04,0.010000,active
9,SEI-USDT,SEI/USDT,1.577249e+05,36.630037,74.112288,1.000000e-04,0.010000,active


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [4]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=ARB%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/candles?symbol=ALGO%2FUSDC&resolution=5&countBack=288&firstDataRequest=1 failed on attempt 1/3: HTTP Error 502: Bad Gateway
GET https://api.nonkyc.io/api/v2/market/orderbook?symbol=BDX%2FUSDT&limit=200 failed on attempt 1/3: HTTP Error 502: Bad Gateway


,enriched_rows,selected_rows,pass_rate
0,343,7,0.020408


,trading_pair,screen_score,passed_filters,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,LTC-USDT,91.606933,True,3.265157e+05,1.872484,10.680000,10.680000,200,293.722958,288,1.000000,0.000000,19.184993,0.130172,
1,SOL-USDT,88.707729,True,7.936260e+05,79.760718,117.482400,0.000000,200,2.967194,288,1.000000,0.000000,31.866698,0.046798,
2,BTC-USDT,87.821561,True,3.512106e+06,50.061071,177.448887,0.000000,200,3.593052,288,1.000000,0.000000,15.401257,0.001048,
3,USDC-USDT,87.808015,True,1.474665e+06,49.002450,6.164760,0.000000,200,4.711801,288,1.000000,0.000000,16.867599,0.000000,
4,NKYC-USDT,87.597490,True,1.498705e+05,14.371827,43.755917,43.755917,200,6.655446,288,1.000000,0.000000,17.642543,0.037575,
5,ETH-USDT,87.222908,True,1.617583e+06,73.666796,5880.338439,0.000000,200,11.260807,288,1.000000,0.000000,21.518058,0.010569,
6,BNB-USDT,85.645303,True,5.804555e+05,50.174761,12.849356,0.000000,200,23.848498,288,1.000000,0.000000,18.390202,0.042607,
7,XMR-USDT,91.011131,False,1.501422e+06,29.753982,0.637620,0.000000,200,2.742423,288,1.000000,0.000000,26.481623,0.082652,top_of_book_quote<5
8,ALGO-USDC,85.424265,False,4.767034e+05,56.657224,0.190806,0.000000,200,2.451551,288,1.000000,0.000000,60.486229,0.132645,top_of_book_quote<5
9,BTC-USDC,84.613176,False,5.291214e+05,69.265391,0.600629,0.000000,200,6.534121,288,1.000000,0.000000,20.416727,0.000231,top_of_book_quote<5


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: {len(passed)} | Rejected: {len(rejected)}


,trading_pair,screen_score,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,natr_bps_mean,efficiency_ratio
0,LTC-USDT,91.606933,3.265157e+05,1.872484,10.680000,10.680000,200,293.722958,19.184993,0.130172
1,SOL-USDT,88.707729,7.936260e+05,79.760718,117.482400,0.000000,200,2.967194,31.866698,0.046798
2,BTC-USDT,87.821561,3.512106e+06,50.061071,177.448887,0.000000,200,3.593052,15.401257,0.001048
3,USDC-USDT,87.808015,1.474665e+06,49.002450,6.164760,0.000000,200,4.711801,16.867599,0.000000
4,NKYC-USDT,87.597490,1.498705e+05,14.371827,43.755917,43.755917,200,6.655446,17.642543,0.037575
5,ETH-USDT,87.222908,1.617583e+06,73.666796,5880.338439,0.000000,200,11.260807,21.518058,0.010569
6,BNB-USDT,85.645303,5.804555e+05,50.174761,12.849356,0.000000,200,23.848498,18.390202,0.042607


,count
rejection_reason,
quote_volume_24h<50000,304
top_of_book_quote<5,303
coverage_ratio<0.90,199
spread_bps>120,41
natr_bps_mean<12,26
natr_bps_mean>400,24
last_trade_age_sec>3600,5
zero_volume_fraction>0.30,3
missing_spread_bps,3


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [6]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=NONKYC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/candle_ingestor_mani...
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/nonkyc/20260403_224348/exchange_rules_patch...


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
LTC-USDT
SOL-USDT
BTC-USDT
USDC-USDT
NKYC-USDT
ETH-USDT
BNB-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  nonkyc:
    enabled: true
    base_url: https://api.nonkyc.io/api/v2
    pairs:
    - LTC/USDT
    - SOL/USDT
    - BTC/USDT
    - USDC/USDT
    - NKYC/USDT
    - ETH/USDT
    - BNB/USDT
    intervals:
    - 5m
    trades:
      enabled: true
      limit: 500
      update_recent_candles: false
      recent_window_minutes: 120


Exchange rules patch (estimates only)
--------------------------------------------------------------------------------
connectors:
  nonkyc:
    pairs:
      LTC-USDT:
        price_tick: 0.